# Heston Model and Synthetic Calibration

This notebook simulates Heston paths, prices a vanilla option and calibrates back to synthetic option prices generated from known parameters.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

import numpy as np
import pandas as pd

from derivatives_engine.calibration.heston_calibration import calibrate_heston, generate_synthetic_heston_chain, parameter_comparison_table
from derivatives_engine.models.black_scholes import call_price
from derivatives_engine.models.heston import HestonParams, heston_price_cf, price_heston_mc, simulate_heston_paths
from derivatives_engine.utils.plotting import plot_paths

In [ ]:
S, K, T, r, q = 100.0, 100.0, 1.0, 0.03, 0.0
params = HestonParams(v0=0.04, kappa=1.5, theta=0.04, sigma_v=0.35, rho=-0.6)
simulation = simulate_heston_paths(S, T, r, q, params, n_paths=500, n_steps=126, seed=4)
plot_paths(simulation.time_grid, simulation.spot_paths, 'Heston spot paths', max_paths=25)

In [ ]:
mc = price_heston_mc(S, K, T, r, q, params, 'call', n_paths=30_000, n_steps=126, seed=5)
cf = heston_price_cf(S, K, T, r, q, params, 'call')
bs = call_price(S, K, T, r, q, params.v0 ** 0.5)
pd.DataFrame([{'heston_mc': mc.price, 'mc_standard_error': mc.standard_error, 'heston_cf': cf, 'black_scholes_same_initial_vol': bs}])

In [ ]:
true_params = HestonParams(v0=0.04, kappa=1.4, theta=0.04, sigma_v=0.35, rho=-0.55)
chain = generate_synthetic_heston_chain(S, r, q, true_params, strikes=np.array([85.0, 100.0, 115.0]), maturities=np.array([0.5, 1.0, 1.5]))
result = calibrate_heston(chain, initial_params=HestonParams(v0=0.045, kappa=1.2, theta=0.045, sigma_v=0.40, rho=-0.45), max_nfev=50)
parameter_comparison_table(true_params, result.params)

In [ ]:
pd.DataFrame([{'rmse': result.rmse, 'mae': result.mae, 'success': result.success, 'n_evaluations': result.n_evaluations}])